# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [6]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5054 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [7]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, list):
                print(f"Lista w argumentach: {len(arg)} elementów")
        for key, val in kwargs.items():
            if isinstance(val, list):
                print(f"Lista '{key}' w argumentach: {len(val)} elementów")
        return func(*args, **kwargs)
    return wrapper

@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

process_data([1, 2, 3, 4, 5], "test")
process_data([10, 20], name="dane", )

Lista w argumentach: 5 elementów
Przetwarzanie test
Lista w argumentach: 2 elementów
Przetwarzanie dane


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [8]:
import functools
from datetime import datetime
import time

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = func(*args, **kwargs)
            duration = time.perf_counter() - start

            log_line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {func.__name__} - czas: {duration:.4f}s\n"
            with open(filename, "a") as f:
                f.write(log_line)

            return result
        return wrapper
    return decorator

@logger("app.log")
def slow_task():
    time.sleep(0.3)
    print("Zadanie wykonane.")

@logger("app.log")
def fast_task(x, y):
    return x + y

slow_task()
print(f"Wynik: {fast_task(2, 3)}")

with open("app.log") as f:
    print("\nZawartość pliku app.log:")
    print(f.read())

Zadanie wykonane.
Wynik: 5

Zawartość pliku app.log:
[2026-03-21 21:19:57] slow_task - czas: 0.3054s
[2026-03-21 21:19:57] fast_task - czas: 0.0000s
[2026-03-21 21:25:37] slow_task - czas: 0.3006s
[2026-03-21 21:25:37] fast_task - czas: 0.0000s



--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [9]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [10]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

class Logger:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self.name, None)
        logging.info(f"[GET] Odczyt '{self.name}' -> {value}")
        return value

    def __set__(self, instance, value):
        logging.info(f"[SET] Zapis '{self.name}' <- {value}")
        instance.__dict__[self.name] = value

class Uzytkownik:
    imie = Logger()
    wiek = Logger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

u = Uzytkownik("Anna", 25)
print(u.imie)
u.wiek = 26
print(u.wiek)

[SET] Zapis 'imie' <- Anna
[SET] Zapis 'wiek' <- 25
[GET] Odczyt 'imie' -> Anna
[SET] Zapis 'wiek' <- 26
[GET] Odczyt 'wiek' -> 26


Anna
26


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [11]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [12]:
def collatz_generator(n):
    while n != 1:
        yield n
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
    yield 1

for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [15]:
current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(
                    f"Wymagana rola: '{role}', aktualna: '{current_user.get('role')}'"
                )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("superuser")
def delete_database():
    print("Baza danych usunięta!")

@require_role("moderator")
def ban_user(name):
    print(f"Użytkownik {name} zbanowany.")

delete_database()

try:
    ban_user("Jan")
except PermissionError as e:
    print(f"Brak dostępu: {e}")

Baza danych usunięta!
Brak dostępu: Wymagana rola: 'moderator', aktualna: 'superuser'


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [16]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"'{self.name}' wymaga typu {self.expected_type.__name__}, "
                f"otrzymano {type(value).__name__}"
            )
        instance.__dict__[self.name] = value

class Produkt:
    nazwa = Typed(str)
    cena = Typed(int)

    def __init__(self, nazwa, cena):
        self.nazwa = nazwa
        self.cena = cena

p = Produkt("Kawa", 15)
print(f"{p.nazwa}: {p.cena} zł")

try:
    p.cena = "tanio"
except TypeError as e:
    print(f"Błąd: {e}")

try:
    Produkt(123, 10)
except TypeError as e:
    print(f"Błąd: {e}")

Kawa: 15 zł
Błąd: 'cena' wymaga typu int, otrzymano str
Błąd: 'nazwa' wymaga typu str, otrzymano int


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [17]:
def prime_generator():
    n = 2
    while True:
        if all(n % i != 0 for i in range(2, int(n**0.5) + 1)):
            yield n
        n += 1

primes_ending_7 = (p for p in prime_generator() if p % 10 == 7)

print("Pierwsze 15 liczb pierwszych kończących się na 7:")
for i, p in enumerate(primes_ending_7, 1):
    if i > 15:
        break
    print(f"  {i}. {p}")

Pierwsze 15 liczb pierwszych kończących się na 7:
  1. 7
  2. 17
  3. 37
  4. 47
  5. 67
  6. 97
  7. 107
  8. 127
  9. 137
  10. 157
  11. 167
  12. 197
  13. 227
  14. 257
  15. 277
